# Buzzlytics — YOLOv8 Fine-Tuning (Google Colab)

Fine-tunes **YOLOv8n** on the unified 4-class bee dataset: `bee`, `pollen_bee`, `varroa_bee`, `wasp`.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is free and enough).
2. Set `ROBOFLOW_API_KEY` in the first code cell.

The notebook rebuilds the exact merged dataset (Andrew Hofer bees + Vespa orientalis hornet, remapped to the 4 classes) using the repo's `datasets/prepare_dataset.py`, so training is fully reproducible.

**Note on class balance:** the merged set is imbalanced — varroa and bee are common, **pollen is sparse (~110 instances)** and wasp moderate (~1000). Expect lower mAP on `pollen_bee`; this is documented as a known limitation in the report.

In [ ]:
# === 0. Configuration ===
ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"  # roboflow.com -> settings -> API
EPOCHS = 100
IMGSZ = 640
BATCH = 16
MODEL = "yolov8n.pt"   # nano: justified by the 10-FPS requirement

assert ROBOFLOW_API_KEY != "PASTE_YOUR_KEY_HERE", "Set your Roboflow API key first"

In [ ]:
# === 1. Confirm GPU ===
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch Runtime -> GPU!)")

In [ ]:
# === 2. Clone repo + install deps ===
%cd /content
![ -d Buzzlytics---CSCI435 ] || git clone https://github.com/okayxsh/Buzzlytics---CSCI435.git
%cd /content/Buzzlytics---CSCI435
!pip -q install "ultralytics==8.4.71" "roboflow==1.1.28" pyyaml

In [ ]:
# === 3. Rebuild the merged 4-class dataset (reproducible) ===
# Pass the key explicitly via --api-key so it works regardless of how the
# Colab shell subprocess inherits environment variables.
!python datasets/prepare_dataset.py --api-key {ROBOFLOW_API_KEY}

# Sanity check: class distribution across all splits (0=bee 1=pollen 2=varroa 3=wasp)
from pathlib import Path
from collections import Counter
c = Counter()
for txt in Path("datasets/data").rglob("labels/*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip():
            c[int(line.split()[0])] += 1
print("instances per class:", dict(sorted(c.items())))
for s in ["train", "valid", "test"]:
    n = len(list(Path(f"datasets/data/{s}/images").glob("*"))) if Path(f"datasets/data/{s}/images").is_dir() else 0
    print(f"{s}: {n} images")

In [ ]:
# === 4. Write a Colab-absolute dataset yaml ===
import yaml
data_cfg = {
    "path": "/content/Buzzlytics---CSCI435/datasets/data",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 4,
    "names": {0: "bee", 1: "pollen_bee", 2: "varroa_bee", 3: "wasp"},
}
with open("/content/bee_dataset.yaml", "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open("/content/bee_dataset.yaml").read())

In [ ]:
# === 5. Fine-tune YOLOv8n ===
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data="/content/bee_dataset.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name="bee_detector",
    patience=25,          # early stop if no val improvement
    seed=42,
    # augmentations (report these in the write-up):
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0, degrees=5.0,
)

In [ ]:
# === 6. Validate + per-class metrics (for the report's results table) ===
best = "runs/detect/bee_detector/weights/best.pt"
metrics = YOLO(best).val(data="/content/bee_dataset.yaml", split="test")
print("\nmAP@50:     ", round(float(metrics.box.map50), 4))
print("mAP@50-95:  ", round(float(metrics.box.map), 4))
names = {0: "bee", 1: "pollen_bee", 2: "varroa_bee", 3: "wasp"}
print("\nPer-class mAP@50:")
for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"  {names.get(int(i), i):<12} {float(ap):.4f}")

In [ ]:
# === 7. Download best.pt ===
# Place the downloaded file at  cv_pipeline/weights/best.pt  in your local repo.
from google.colab import files
files.download("runs/detect/bee_detector/weights/best.pt")

# Also save the training plots (confusion matrix, PR curves) for the report:
!ls runs/detect/bee_detector/*.png
# files.download('runs/detect/bee_detector/confusion_matrix.png')  # uncomment to grab